# Phase 0 — Lesson 10: Terminal & Shell
> The terminal is where AI engineers live. Get comfortable here.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 10
**Type:** Learn | **Languages:** -- | **Time:** ~35 minutes
**Prerequisites:** Phase 0, Lesson 01

This lesson explains *why* each terminal tool matters for AI work, not just the commands. Run the code cells to see visualizations of tmux sessions, piping, and remote workflows.


# The Problem: You'll Live in the Terminal

---


You will spend more time in the terminal than in any editor. Training runs, GPU monitoring, log tailing, remote SSH sessions, environment management. Every AI workflow touches the shell. If you're slow here, you're slow everywhere.

This lesson covers the terminal skills that matter for AI work. No history of Unix. No deep-dive into Bash scripting. Just what you need.


# Visual: tmux Session — Three Panes, One Terminal

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Pane 1: Training
rect_p1 = patches.Rectangle((0.1, 0.35), 0.4, 0.5, facecolor='#3498db', edgecolor='black', linewidth=2)
ax.add_patch(rect_p1)
ax.text(0.3, 0.7, 'Pane 1: Training', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.3, 0.55, 'python train.py', ha='center', va='center', fontsize=9, color='white')
ax.text(0.3, 0.45, 'Epoch 12/100...', ha='center', va='center', fontsize=9, color='white')

# Pane 2: GPU monitor
rect_p2 = patches.Rectangle((0.5, 0.35), 0.4, 0.5, facecolor='#2ecc71', edgecolor='black', linewidth=2)
ax.add_patch(rect_p2)
ax.text(0.7, 0.7, 'Pane 2: GPU Monitor', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.7, 0.55, 'watch -n1 nvidia-smi', ha='center', va='center', fontsize=9, color='white')
ax.text(0.7, 0.45, 'GPU: 78% | Mem: 14/24G', ha='center', va='center', fontsize=9, color='white')

# Pane 3: Logs
rect_p3 = patches.Rectangle((0.1, 0.1), 0.8, 0.2, facecolor='#e74c3c', edgecolor='black', linewidth=2)
ax.add_patch(rect_p3)
ax.text(0.5, 0.2, 'Pane 3: Logs', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.5, 0.13, 'tail -f logs/train.log | grep loss', ha='center', va='center', fontsize=9, color='white')

ax.text(0.5, 0.92, 'Three things running at once. One terminal.', ha='center', fontsize=10, style='italic')
ax.text(0.5, 0.02, 'Detach, go home, SSH back in, reattach — training keeps running.', ha='center', fontsize=9, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("tmux: Training, Monitoring, and Logging Simultaneously", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **tmux** lets you detach from a session and reattach later. Training keeps running even when you close your laptop.


# Step 1: Know Your Shell

---


Check which shell you're running:

```bash
echo $SHELL
```

Most systems use `bash` or `zsh`. Both work fine. The commands in this course work in either.

**Essentials:**

```bash
# Move around
cd ~/projects/ai-engineering-from-scratch
pwd
ls -la

# History search (most useful shortcut you'll learn)
# Ctrl+R then type part of a previous command
# Press Ctrl+R again to cycle through matches

# Clear terminal
clear   # or Ctrl+L

# Cancel a running command
# Ctrl+C

# Suspend a running command (resume with fg)
# Ctrl+Z
```


# Step 2: Piping and Redirects

---


Piping connects commands together. This is how you process logs, filter output, and chain tools. You will use this constantly.

```bash
# Count how many times 'loss' appears in a log
cat train.log | grep "loss" | wc -l

# Extract just the loss values from training output
grep "loss:" train.log | awk '{print $NF}' > losses.txt

# Watch a log file update in real time, filtering for errors
tail -f train.log | grep --line-buffered "ERROR"

# Sort experiments by final accuracy
grep "final_accuracy" results/*.log | sort -t= -k2 -n -r

# Redirect stdout and stderr to separate files
python train.py > output.log 2> errors.log

# Redirect both to the same file
python train.py > train_full.log 2>&1
```

| Symbol | What it does |
|--------|-------------|
| `>` | Write stdout to file (overwrite) |
| `>>` | Append stdout to file |
| `2>` | Write stderr to file |
| `2>&1` | Send stderr to same place as stdout |
| `\|` | Send stdout of one command as stdin to the next |


# Visual: How a Pipe Chain Works

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

steps = [
    ('cat train.log', 'Raw log file', '#2c3e50'),
    ('grep "loss"', 'Filter lines with loss', '#34495e'),
    ('awk', 'Extract just the number', '#3498db'),
    ('> losses.txt', 'Save to file', '#2ecc71'),
]

x = 0.1
for label, desc, color in steps:
    rect = patches.Rectangle((x, 0.35), 0.2, 0.4, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 0.1, 0.6, label, ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    ax.text(x + 0.1, 0.25, desc, ha='center', va='center', fontsize=7, style='italic')
    x += 0.22

for i in range(len(steps)-1):
    ax.annotate('', xy=(x-0.22+0.2, 0.55), xytext=(x-0.22+0.2, 0.5),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    x -= 0.22

ax.text(0.5, 0.1, 'Each command transforms the data. The pipe passes it forward.', ha='center', fontsize=10, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Pipe Chain: cat → grep → awk → file", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Pipes are how AI engineers process logs.** You'll filter training output constantly — looking for loss values, errors, or specific patterns.


# Step 3: Background Processes

---


Training runs take hours. You don't want to keep your terminal open the whole time.

```bash
# Run in background (output still goes to terminal)
python train.py &

# Run in background, immune to hangup (closing terminal won't kill it)
nohup python train.py > train.log 2>&1 &

# Check what's running in background
jobs
ps aux | grep train.py

# Bring a background job to foreground
fg %1

# Kill a background process
kill %1
# or find its PID and kill that
kill $(pgrep -f "train.py")
```

| Method | Survives terminal close? | Can reattach? |
|--------|-------------------------|---------------|
| `command &` | No | No |
| `nohup command &` | Yes | No (check log file) |
| `screen` / `tmux` | Yes | Yes |

For anything longer than a few minutes, use tmux.


# Step 4: tmux

---


tmux lets you create persistent terminal sessions with multiple panes. This is the single most useful tool for managing training runs.

```bash
# Install
# macOS
brew install tmux
# Ubuntu
sudo apt install tmux

# Start a named session
tmux new -s training

# Split horizontally
# Ctrl+B then "

# Split vertically
# Ctrl+B then %

# Navigate between panes
# Ctrl+B then arrow keys

# Detach (session keeps running)
# Ctrl+B then d

# Reattach
tmux attach -t training

# List sessions
tmux ls

# Kill a session
tmux kill-session -t training
```

**A typical AI workflow session:**
```bash
tmux new -s train

# Pane 1: start training
python train.py --epochs 100 --lr 1e-4

# Ctrl+B, " to split, then run GPU monitor
watch -n1 nvidia-smi

# Ctrl+B, % to split vertically, tail the logs
tail -f logs/experiment.log

# Now detach with Ctrl+B, d
# SSH out, go get coffee, come back
# tmux attach -t train
```


# Step 5: Monitoring with htop and nvtop

---


```bash
# System processes (better than top)
htop

# GPU processes (if you have NVIDIA GPU)
# Install: sudo apt install nvtop (Ubuntu) or brew install nvtop (macOS)
nvtop

# Quick GPU check without nvtop
nvidia-smi

# Watch GPU usage update every second
watch -n1 nvidia-smi

# See which processes are using the GPU
nvidia-smi --query-compute-apps=pid,name,used_memory --format=csv
```

**htop keybindings you'll use:**
- `F6` or `>` to sort by column (sort by memory to find memory leaks)
- `F5` to toggle tree view (see child processes)
- `F9` to kill a process
- `/` to search for a process name


# Visual: Monitoring Tools by Purpose

---


In [ ]:
import matplotlib.pyplot as plt

tools = ['htop', 'nvtop', 'nvidia-smi', 'watch']
purposes = ['System processes', 'GPU processes', 'Quick GPU check', 'Repeated command']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

for i, (tool, purpose, color) in enumerate(zip(tools, purposes, colors)):
    rect = plt.Rectangle((0.1, 0.75 - i*0.18), 0.2, 0.14, facecolor=color, edgecolor='black', linewidth=2)
    plt.gca().add_patch(rect)
    plt.text(0.22, 0.75 - i*0.18 + 0.07, tool, ha='left', va='center', color='white', fontweight='bold', fontsize=11)
    plt.text(0.45, 0.75 - i*0.18 + 0.07, purpose, ha='left', va='center', fontsize=9)

plt.text(0.5, 0.05, 'Keep one terminal for each tool. Split them with tmux.', ha='center', fontsize=10, style='italic')

plt.xlim(0, 1)
plt.ylim(0, 1)
plt.title('Monitoring Tools: Use the Right One for the Job', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


> **nvtop** is like htop but for GPUs — it shows which processes are using GPU memory. Essential for debugging OOM (out of memory) errors.


# Step 6: SSH for Remote GPU Boxes

---


When you rent a cloud GPU (Lambda, RunPod, Vast.ai), you connect via SSH.

```bash
# Basic connection
ssh user@gpu-box-ip

# With a specific key
ssh -i ~/.ssh/my_gpu_key user@gpu-box-ip

# Copy files to remote
scp model.pt user@gpu-box-ip:~/models/

# Copy files from remote
scp user@gpu-box-ip:~/results/metrics.json ./

# Sync a whole directory (faster for many files)
rsync -avz ./data/ user@gpu-box-ip:~/data/

# Port forward (access remote Jupyter/TensorBoard locally)
ssh -L 8888:localhost:8888 user@gpu-box-ip
# Now open localhost:8888 in your browser

# SSH config for convenience
# Add to ~/.ssh/config:
# Host gpu
#     HostName 192.168.1.100
#     User ubuntu
#     IdentityFile ~/.ssh/gpu_key
#
# Then just:
# ssh gpu
```


# Visual: SSH Workflow for Remote GPU Work

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Local
rect_local = patches.Rectangle((0.1, 0.3), 0.35, 0.5, facecolor='#3498db', edgecolor='black', linewidth=2)
ax.add_patch(rect_local)
ax.text(0.275, 0.65, 'LOCAL MACHINE', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.275, 0.5, 'Edit code, run Jupyter\nvia port forwarding', ha='center', va='center', fontsize=8, color='white')

# Remote GPU box
rect_remote = patches.Rectangle((0.55, 0.3), 0.35, 0.5, facecolor='#e74c3c', edgecolor='black', linewidth=2)
ax.add_patch(rect_remote)
ax.text(0.725, 0.65, 'REMOTE GPU BOX', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.725, 0.5, 'Training runs here\nGPU does the work', ha='center', va='center', fontsize=8, color='white')

# Arrows
ax.annotate('', xy=(0.46, 0.55), xytext=(0.44, 0.55),
            arrowprops=dict(arrowstyle='->', color='#f39c12', lw=2))
ax.text(0.5, 0.58, 'scp / rsync\ncode & data up', ha='center', fontsize=8, color='#f39c12')

ax.annotate('', xy=(0.46, 0.45), xytext=(0.44, 0.45),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
ax.text(0.5, 0.42, 'scp / rsync\nresults down', ha='center', fontsize=8, color='#27ae60')

ax.annotate('', xy=(0.46, 0.35), xytext=(0.44, 0.35),
            arrowprops=dict(arrowstyle='->', color='#3498db', lw=2))
ax.text(0.5, 0.32, 'ssh -L 8888:localhost:8888\nJupyter locally', ha='center', fontsize=8, color='#3498db')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("SSH Workflow: Code Up, Results Down, Jupyter Forwarded", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Use `rsync` over `scp` for anything large.** It only transfers changed bytes and handles interrupted connections.


# Step 7: Useful Aliases for AI Work

---


Add these to your `~/.bashrc` or `~/.zshrc`:

```bash
# GPU status at a glance
alias gpu='nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv,noheader'

# Kill all Python training processes
alias killtraining='pkill -f "python.*train"'

# Quick virtual environment activate
alias ae='source .venv/bin/activate'

# Watch training loss
alias watchloss='tail -f logs/*.log | grep --line-buffered "loss"'
```

See `code/shell_aliases.sh` for the full set.


# Step 8: Common AI Terminal Patterns

---


These come up repeatedly in practice:

```bash
# Run training, log everything, notify when done
python train.py 2>&1 | tee train.log; echo "DONE" | mail -s "Training complete" you@email.com

# Compare two experiment logs side by side
diff <(grep "accuracy" exp1.log) <(grep "accuracy" exp2.log)

# Find the largest model files (clean up disk space)
find . -name "*.pt" -o -name "*.safetensors" | xargs du -h | sort -rh | head -20

# Download a model from Hugging Face
wget https://huggingface.co/model/resolve/main/model.safetensors

# Untar a dataset
tar xzf dataset.tar.gz -C ./data/

# Count lines in all Python files
find . -name "*.py" | xargs wc -l | tail -1

# Check disk space (training data fills disks fast)
df -h
du -sh ./data/*

# Environment variable check before training
env | grep -i cuda
env | grep -i torch
```


# Use It: When Each Tool Comes Into Play

---


| Tool | When you use it |
|------|----------------|
| tmux | Every training run (Phases 3+) |
| `tail -f` + `grep` | Monitoring training logs |
| `nohup` / `&` | Quick background tasks |
| `htop` / `nvtop` | Debugging slow training, OOM errors |
| SSH + `rsync` | Working on cloud GPUs |
| Piping + redirects | Processing experiment results |
| Aliases | Saving time on repetitive commands |


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** What does the pipe operator `|` do in a shell command?
> **Explanation:** Sends the standard output of one command as standard input to the next. For example, `cat log.txt | grep error` sends the contents of log.txt as input to grep.

**Q2:** What happens when you close the terminal that started a process?
> **Explanation:** The process receives a hangup signal (SIGHUP) and typically terminates. Tools like tmux, nohup, or screen prevent this.

**Q3:** What is the key advantage of tmux over `nohup command &`?
> **Explanation:** tmux lets you detach, reattach, and see live output with multiple panes. nohup only logs to a file with no way to interact or reattach.

**Q4:** What does `python train.py > output.log 2>&1` accomplish?
> **Explanation:** Redirects both standard output and standard error to output.log. All output and errors are captured in one file.

**Q5:** Which command lets you access a remote Jupyter notebook on port 8888?
> **Explanation:** `ssh -L 8888:localhost:8888 user@gpu-box`. SSH local port forwarding maps a remote port to your local machine.


# Practice Exercises

---


Do these in order:

**Exercise 1:** Install tmux, create a session with three panes, and run `htop` in one, `watch -n1 date` in another, and a Python script in the third. Detach and reattach.

**Exercise 2:** Add the aliases from `code/shell_aliases.sh` to your shell config and reload.

**Exercise 3:** Create a fake training log and use `grep`, `tail`, and `awk` to extract loss values.
```bash
for i in $(seq 1 100); do echo "epoch $i loss: $(echo "scale=4; 1/$i" | bc)"; sleep 0.1; done > fake_train.log
```

**Exercise 4:** Set up an SSH config entry for a server you have access to (or use `localhost` to practice).


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Shell | "The terminal" | The program that interprets your commands (bash, zsh, fish) |
| tmux | "Terminal multiplexer" | A program that lets you run multiple terminal sessions inside one window, and detach/reattach |
| Pipe | "The bar thing" | The `\|` operator that sends one command's output as input to another |
| PID | "Process ID" | A unique number assigned to every running process, used to monitor or kill it |
| nohup | "No hangup" | Runs a command immune to the hangup signal, so closing the terminal won't kill it |
| SSH | "Connecting to the server" | Secure Shell, an encrypted protocol for running commands on a remote machine |


# Next Steps

---


When you're comfortable in the terminal, continue to:
**Lesson 11: Linux for AI**

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
